# Decision tree for classification

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

In this notebook, we train a decision tree to predict which clients subscribe to a term deposit, and then we tune the hyperparameters that control the growth of the tree with a random search.

In [1]:
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.tree import DecisionTreeClassifier

## Load data

We use the bank marketing data that we prepared in the notebook of the folder prepare-data.

In [2]:
df = pd.read_csv("../prepare-data/bank_marketing.csv")
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,NaN,0,unknown,0
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,NaN,0,unknown,0
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,NaN,0,unknown,0
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,NaN,0,unknown,0
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,NaN,0,unknown,0


### Categorical variables

Because decision trees sort the values of the features in increasing order and then test every possible cut-off point, encoding categorical variables with integers (that is,
replacing the name of the categories with numbers) is actually a good idea. The opposite is true for linear models, where one hot encoding is a better approach.

### Missing values

For the same reason, missing values are not a problem for decision trees. Scikit-learn sends all the rows with a missing value to whichever side of the split reduces the
impurity more, so we can leave them as they are, which is what we do with `pdays`. An alternative that works with any implementation is to replace them with an arbitrary value
outside the distribution, for example -1 for `pdays`, since all the values in that variable are positive.

> **Note:** the native support for missing values is recent. Scikit-learn added it to `DecisionTreeClassifier` and `DecisionTreeRegressor` in version 1.3, and to
> `RandomForestClassifier` and `RandomForestRegressor` in version 1.4.

In [3]:
# the trees from Scikit-learn take numbers, so we replace the categories by integers
categorical_vars = df.select_dtypes("str").columns
df[categorical_vars] = df[categorical_vars].apply(lambda var: var.astype("category").cat.codes)

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,4,1,2,0,2143,1,0,2,5,8,261,1,NaN,0,3,0
1,44,9,2,1,0,29,1,0,2,5,8,151,1,NaN,0,3,0
2,33,2,1,1,0,2,1,1,2,5,8,76,1,NaN,0,3,0
3,47,1,1,3,0,1506,1,0,2,5,8,92,1,NaN,0,3,0
4,33,11,2,3,0,1,0,0,2,5,8,198,1,NaN,0,3,0


We split the data and use one portion to train the model and another portion to evaluate it.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns="y"),
    df["y"],
    test_size=0.3,
    stratify=df["y"],
    random_state=0,
)

X_train.shape, X_test.shape

((31647, 16), (13564, 16))

## Set up the model

We show the hyperparameters with their default values, so that we can see the effect of changing them later on. 

The default values grow the tree without early stopping or pruning.

In [5]:
tree = DecisionTreeClassifier(
    criterion="gini",  # the loss function
    splitter="best",   # finds best split
    max_features=None, # evaluates all features at each node
    random_state=0,    # reproducibility
    
    # how far the tree grows.
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,

    # pruning
    ccp_alpha=0.0,
)

tree.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",0
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

Decision trees are deterministic by definition. So why do we need a random state?

When two splits reduce the impurity by exactly the same amount, the one that wins is the feature that scikit-learn sees first. And since it shuffles the features before evaluating, the random-state guarantees the same result. So worth only when there are 2 or more splits that give the same purity gain. It can happen :) 


The `random_state` matters more when we set `max_features` to a value smaller than the total number of features, which is unlikely if we are fitting just a decision tree.

## Evaluate the model

We look at the balanced accuracy and at the roc-auc in the train and test sets to find out if the tree overfits.

In [6]:
def evaluate(model, X_train, y_train, X_test, y_test):
    for name, X, y in [("train", X_train, y_train), ("test", X_test, y_test)]:
        balanced_accuracy = balanced_accuracy_score(y, model.predict(X))
        roc_auc = roc_auc_score(y, model.predict_proba(X)[:, 1])
        print(f"{name} balanced accuracy: {balanced_accuracy:.4f}, roc-auc: {roc_auc:.4f}")


evaluate(tree, X_train, y_train, X_test, y_test)

train balanced accuracy: 1.0000, roc-auc: 1.0000
test balanced accuracy: 0.6985, roc-auc: 0.6985


The model clearly overfits the data.

## Search for better hyperparameters

The tree that we trained so far grows until the leaves are pure, and that is why it overfits. Now we sample values for the hyperparameters that control the growth of the tree from continuous and discrete distributions, and we evaluate each combination with cross validation.

In [7]:
param_distributions = {
    "max_depth": randint(2, 10),
    # as we are using floats, these represent fraction of
    # the training set
    "min_samples_split": uniform(0.001, 0.1),
    "min_samples_leaf": uniform(0.001, 0.1),
    "min_impurity_decrease": loguniform(1e-6, 1e-2),
    # for pruning
    # "ccp_alpha": loguniform(1e-6, 1e-2),
}

search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=0),
    param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=0,
)

search.fit(X_train, y_train)

search.best_params_

{'max_depth': 8,
 'min_impurity_decrease': np.float64(2.559642090152666e-06),
 'min_samples_leaf': np.float64(0.021887675609483473),
 'min_samples_split': np.float64(0.01713095178849963)}

### My 2 cent

You need to play a bit with the hyperparameters to get a feeling. 

I personally find that controlling the growth is faster and it returns good results. Pruning takes a lot more time, and it doesn't necessarily return better trees. So I do not use the alpha parameter in practice.

For this particular example, if we do not change alpha, then we find the best tree after 20 samples with random search. If we instead use alpha, we need to sample 60 trees to find the best one.

In [15]:
# the percentage of missing data in the variables that show missing values
missing = df.isna().mean().mul(100)
missing[missing > 0].round(2)

pdays    81.74
dtype: float64

In [8]:
# the standard deviation tells us how much the metric changes across the folds
results = pd.DataFrame(search.cv_results_)
results = results.sort_values("rank_test_score")

results[["mean_test_score", "std_test_score", "rank_test_score"]].head(10)

,mean_test_score,std_test_score,rank_test_score
15,0.878330,0.006475,1
10,0.872356,0.006968,2
14,0.864433,0.002914,3
2,0.864389,0.008686,4
16,0.852028,0.004994,5
13,0.851771,0.005243,6
8,0.851683,0.006805,7
4,0.844948,0.005834,8
1,0.842382,0.007228,9
3,0.842108,0.006573,10


In [9]:
best = results.iloc[0]

print(f"best cross validation roc-auc: {best['mean_test_score']:.4f} +/- {best['std_test_score']:.4f}")

best cross validation roc-auc: 0.8783 +/- 0.0065


### Evaluate the new model

We compare the performance of the tuned tree with the performance of the tree that we trained with the default hyperparameters.

In [10]:
evaluate(search.best_estimator_, X_train, y_train, X_test, y_test)

train balanced accuracy: 0.6482, roc-auc: 0.8854
test balanced accuracy: 0.6478, roc-auc: 0.8807


In [11]:
search.best_estimator_.predict_proba(X_test)

array([[0.98588235, 0.01411765],
       [0.87442713, 0.12557287],
       [0.99871366, 0.00128634],
       ...,
       [0.96825397, 0.03174603],
       [0.99871366, 0.00128634],
       [0.75340909, 0.24659091]], shape=(13564, 2))

In [12]:
import numpy as np
np.unique(search.best_estimator_.predict_proba(X_test)[:, 1])

array([0.0007716 , 0.00128634, 0.00397417, 0.00409626, 0.01098901,
       0.01411765, 0.0180791 , 0.01920439, 0.02621232, 0.03174603,
       0.03477051, 0.03504929, 0.04084158, 0.04232804, 0.05270458,
       0.05307263, 0.07183908, 0.07285546, 0.09115646, 0.0926143 ,
       0.12557287, 0.1488498 , 0.17422096, 0.24659091, 0.25238745,
       0.25944584, 0.35323383, 0.36050955, 0.48707483, 0.58862325,
       0.64792503])

In [13]:
len(np.unique(search.best_estimator_.predict_proba(X_test)[:, 1]))

31

In [14]:
search.best_estimator_.predict(X_test)

array([0, 0, 0, ..., 0, 0, 0], shape=(13564,))